In [1]:
from datetime import datetime
import json
import os
from pathlib import Path
import pickle
import sys
import time

import pandas as pd

repo_root = Path.cwd()
while not (repo_root / "data/troupe/TLS/processed_data/trees.pkl").is_file():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate the repository data directory")
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

datasets = ("TLS", "TLSC")
states = ("NMPs", "Somite", "NeuralTube")
CACHE_VERSION = 1
REFRESH_CACHE = False  # Set True to force preprocessing and replace the cache.
cache_path = Path.home() / ".cache" / "phylodynamic-carta" / "tls_result_tables.json"

def log_progress(message):
    timestamp = datetime.now().astimezone().isoformat(timespec="seconds")
    print(f"[{timestamp}] {message}", flush=True)

def source_files():
    paths = [
        repo_root / "data/troupe" / dataset / "processed_data/trees.pkl"
        for dataset in datasets
    ]
    for dataset in datasets:
        data_dir = repo_root / "data/troupe" / dataset / "fixed_sampling_0.1"
        paths.extend(path for path in data_dir.glob("inference_transition_times_*") if path.is_file())
    paths.extend((repo_root / "data/troupe").glob("transition-times-*.txt"))
    return sorted(set(paths))

def fingerprint(paths):
    return [
        {
            "path": str(path.relative_to(repo_root)),
            "size": path.stat().st_size,
            "mtime_ns": path.stat().st_mtime_ns,
        }
        for path in paths
    ]

def records(frame):
    return json.loads(frame.to_json(orient="records"))

current_fingerprint = fingerprint(source_files())
cache_payload = None
if REFRESH_CACHE:
    log_progress("Cache refresh requested; preprocessing source data again.")
elif cache_path.is_file():
    log_progress(f"Checking preprocessing cache: {cache_path}")
    saved_cache = json.loads(cache_path.read_text(encoding="utf-8"))
    if (
        saved_cache.get("version") == CACHE_VERSION
        and saved_cache.get("source_fingerprint") == current_fingerprint
    ):
        cache_payload = saved_cache
        log_progress("Source files match; loading preprocessed TLS tables from cache.")
    else:
        log_progress("Cache is stale (source files or cache version changed); preprocessing again.")
else:
    log_progress("No preprocessing cache found; preprocessing source data.")

if cache_payload is not None:
    leaf_summary = pd.DataFrame(cache_payload["leaf_summary"])
    fit_summary = pd.DataFrame(cache_payload["fit_summary"])
    state_parameters = pd.DataFrame(cache_payload["state_parameters"])
else:
    from src.results.tls.leaf_counts import build_leaf_count_rows
    from src.summarize_fixed_sampling_transition_runs import load_runs

    started = time.perf_counter()
    log_progress("Loading original TLS/TLSC tree pickles.")
    trees_by_dataset = {}
    for dataset in datasets:
        tree_path = repo_root / "data/troupe" / dataset / "processed_data/trees.pkl"
        with tree_path.open("rb") as tree_file:
            trees_by_dataset[dataset] = pickle.load(tree_file)

    log_progress("Counting leaf states and three-state coverage.")
    leaf_rows, _ = build_leaf_count_rows(trees_by_dataset, datasets, states)
    leaf_summary_rows = []
    for row in leaf_rows:
        if row["tree_name"] != "TOTAL":
            continue
        selected_leaves = sum(row[state] for state in states)
        leaf_summary_rows.append({
            "dataset": row["dataset"],
            "number_of_trees": len(trees_by_dataset[row["dataset"]]),
            **{state: row[state] for state in states},
            "all_labeled_leaves": row["total_labeled_leaves"],
            "three_state_leaves": selected_leaves,
            "three_state_leaf_percent": 100 * selected_leaves / row["total_labeled_leaves"],
        })
    leaf_summary = pd.DataFrame(leaf_summary_rows)

    log_progress("Parsing transition-count fit outputs, checkpoints, and logs.")
    fit_summary_rows = []
    state_parameter_rows = []
    for dataset in datasets:
        data_dir = repo_root / "data/troupe" / dataset / "fixed_sampling_0.1"
        for fit in load_runs(data_dir):
            count = fit.count
            delta_t = 1.0 / count
            q_s = fit.transition_probabilities.get((0, 1), 0.0) / delta_t
            q_nt = fit.transition_probabilities.get((0, 2), 0.0) / delta_t
            fit_summary_rows.append({
                "dataset": dataset,
                "transition_times": count,
                "nominal_delta_t": delta_t,
                "fit_status": "completed" if fit.completed else "checkpoint/incomplete",
                "last_iteration": fit.iterations_completed,
                "source_file": fit.source.name,
                "q_NMP_to_S": q_s,
                "q_NMP_to_NT": q_nt,
                "q_NT_over_q_S": q_nt / q_s if q_s else float("nan"),
                "b_S_over_b_NT": fit.birth_rates[1] / fit.birth_rates[2],
            })
            for index, state in enumerate(states):
                birth = fit.birth_rates[index]
                death = fit.death_rates[index]
                sampling = fit.sampling_rates[index]
                state_parameter_rows.append({
                    "dataset": dataset,
                    "transition_times": count,
                    "fit_status": "completed" if fit.completed else "checkpoint/incomplete",
                    "state": state,
                    "birth_rate": birth,
                    "death_rate": death,
                    "sampling_probability": sampling,
                    "net_growth_rate": birth - death,
                    "scaled_birth_rate": birth * sampling,
                })
    fit_summary = pd.DataFrame(fit_summary_rows).sort_values(["dataset", "transition_times"])
    state_parameters = pd.DataFrame(state_parameter_rows).sort_values(["dataset", "transition_times", "state"])
    log_progress(f"Preprocessing tables finished in {time.perf_counter() - started:.2f} seconds.")

    cache_payload = {
        "version": CACHE_VERSION,
        "source_fingerprint": fingerprint(source_files()),
        "leaf_summary": records(leaf_summary),
        "fit_summary": records(fit_summary),
        "state_parameters": records(state_parameters),
    }
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_cache = cache_path.with_name(f"{cache_path.name}.{os.getpid()}.tmp")
    temporary_cache.write_text(json.dumps(cache_payload, indent=2), encoding="utf-8")
    temporary_cache.replace(cache_path)
    log_progress(f"Saved preprocessed tables to cache: {cache_path}")

fit_summary = fit_summary.sort_values(["dataset", "transition_times"])
state_parameters = state_parameters.sort_values(["dataset", "transition_times", "state"])

# This is the table of transition-ratio and birth-ratio values for the results text.
with pd.option_context("display.float_format", "{:.2f}".format):
    display(fit_summary)

# Per-state birth, death, net growth, and scaled birth estimates by schedule.
with pd.option_context("display.float_format", "{:.2f}".format):
    display(state_parameters)

# Focused 100-transition comparison; TLS may be a checkpoint rather than a final fit.
with pd.option_context("display.float_format", "{:.2f}".format):
    display(fit_summary.loc[fit_summary["transition_times"] == 100])
    display(state_parameters.loc[state_parameters["transition_times"] == 100])

[2026-10-26T20:41:08-04:00] Checking preprocessing cache: /Users/samin/.cache/phylodynamic-carta/tls_result_tables.json
[2026-10-26T20:41:08-04:00] Source files match; loading preprocessed TLS tables from cache.


,dataset,transition_times,nominal_delta_t,fit_status,last_iteration,source_file,q_NMP_to_S,q_NMP_to_NT,q_NT_over_q_S,b_S_over_b_NT
0,TLS,5,0.20,completed,5430,inference_transition_times_5.out,1.28,1.78,1.39,0.66
1,TLS,10,0.10,completed,25990,inference_transition_times_10.out,2.00,2.70,1.35,0.68
2,TLS,15,0.07,completed,29400,inference_transition_times_15.out,2.41,3.15,1.31,0.67
3,TLS,20,0.05,completed,23210,inference_transition_times_20.out,2.64,3.62,1.37,0.69
4,TLS,25,0.04,completed,21390,inference_transition_times_25.out,2.81,3.80,1.35,0.68
5,TLS,30,0.03,checkpoint/incomplete,18860,inference_transition_times_30.tmp.out,2.97,3.88,1.31,0.67
6,TLS,40,0.03,checkpoint/incomplete,13430,inference_transition_times_40.tmp.out,3.14,4.18,1.33,0.68
7,TLS,50,0.02,checkpoint/incomplete,10440,inference_transition_times_50.tmp.out,3.25,4.29,1.32,0.68
8,TLS,75,0.01,checkpoint/incomplete,6360,inference_transition_times_75.tmp.out,3.40,4.49,1.32,0.68
9,TLS,100,0.01,checkpoint/incomplete,4360,inference_transition_times_100.tmp.out,3.47,4.55,1.31,0.67


,dataset,transition_times,fit_status,state,birth_rate,death_rate,sampling_probability,net_growth_rate,scaled_birth_rate
0,TLS,5,completed,NMPs,8.23,0.78,0.10,7.45,0.82
1,TLS,5,completed,NeuralTube,10.03,8.29,0.10,1.74,1.00
2,TLS,5,completed,Somite,6.62,4.87,0.10,1.75,0.66
3,TLS,10,completed,NMPs,8.82,0.00,0.10,8.82,0.88
4,TLS,10,completed,NeuralTube,10.17,8.82,0.10,1.34,1.02
...,...,...,...,...,...,...,...,...,...
61,TLSC,100,completed,NeuralTube,7.46,7.46,0.10,0.00,0.75
62,TLSC,100,completed,Somite,5.78,2.98,0.10,2.80,0.58
63,TLSC,200,checkpoint/incomplete,NMPs,8.39,1.50,0.10,6.90,0.84
64,TLSC,200,checkpoint/incomplete,NeuralTube,7.45,7.45,0.10,0.00,0.74


,dataset,transition_times,nominal_delta_t,fit_status,last_iteration,source_file,q_NMP_to_S,q_NMP_to_NT,q_NT_over_q_S,b_S_over_b_NT
9,TLS,100,0.01,checkpoint/incomplete,4360,inference_transition_times_100.tmp.out,3.47,4.55,1.31,0.67
20,TLSC,100,0.01,completed,3270,inference_transition_times_100.out,1.66,1.20,0.72,0.78


,dataset,transition_times,fit_status,state,birth_rate,death_rate,sampling_probability,net_growth_rate,scaled_birth_rate
27,TLS,100,checkpoint/incomplete,NMPs,10.36,0.00,0.10,10.36,1.04
28,TLS,100,checkpoint/incomplete,NeuralTube,10.82,9.95,0.10,0.87,1.08
29,TLS,100,checkpoint/incomplete,Somite,7.31,6.63,0.10,0.68,0.73
60,TLSC,100,completed,NMPs,8.44,1.59,0.10,6.85,0.84
61,TLSC,100,completed,NeuralTube,7.46,7.46,0.10,0.00,0.75
62,TLSC,100,completed,Somite,5.78,2.98,0.10,2.80,0.58


## Result text

In [2]:
def fit_for(dataset, count=100):
    rows = fit_summary.loc[
        (fit_summary["dataset"] == dataset)
        & (fit_summary["transition_times"] == count)
    ]
    if len(rows) != 1:
        raise ValueError(f"Expected one {dataset} fit with {count} transition times; found {len(rows)}")
    return rows.iloc[0]

def parameter_for(dataset, state):
    rows = state_parameters.loc[
        (state_parameters["dataset"] == dataset)
        & (state_parameters["transition_times"] == 100)
        & (state_parameters["state"] == state)
    ]
    if len(rows) != 1:
        raise ValueError(f"Expected one 100-transition estimate for {dataset}/{state}")
    return rows.iloc[0]

def leaf_total(dataset):
    rows = leaf_summary.loc[leaf_summary["dataset"] == dataset]
    if len(rows) != 1:
        raise ValueError(f"Expected one leaf-count row for {dataset}")
    return rows.iloc[0]

def num(value):
    return f"{float(value):.2f}"

def integer(value):
    return f"{int(value):,}"

tls_fit = fit_for("TLS")
tlsc_fit = fit_for("TLSC")
tls_leaves = leaf_total("TLS")
tlsc_leaves = leaf_total("TLSC")
tls_nmp = parameter_for("TLS", "NMPs")
tls_s = parameter_for("TLS", "Somite")
tls_nt = parameter_for("TLS", "NeuralTube")
tlsc_nmp = parameter_for("TLSC", "NMPs")
tlsc_s = parameter_for("TLSC", "Somite")
tlsc_nt = parameter_for("TLSC", "NeuralTube")

tls_fit_caveat = (
    f"The TLS 100-transition fit is an incomplete checkpoint at iteration {int(tls_fit['last_iteration']):,}; its estimates are provisional."
    if tls_fit["fit_status"] != "completed"
    else "The TLS 100-transition fit completed."
)
tlsc_fit_status = (
    "The TLS$^{\\mathrm{CL}}$ 100-transition fit completed."
    if tlsc_fit["fit_status"] == "completed"
    else f"The TLS$^{{\\mathrm{{CL}}}}$ 100-transition fit is an incomplete checkpoint at iteration {int(tlsc_fit['last_iteration']):,}."
)

result_text = r'''We applied \\ourmethod to study neuromesodermal progenitor (NMP) dynamics in mouse trunk-like structures (TLS), a stem-cell-derived model of early mammalian trunk development.
NMPs maintain a progenitor population while giving rise to neural tube and somitic cells, making TLS a useful system for investigating the balance between progenitor maintenance and differentiation~\\cite{veenvliet2020mouse,bolondi2024reconstructing,sashittal2026inferring}.
We compared standard TLS with TLS$^{\\mathrm{CL}}$, a perturbed condition treated with the WNT activator CHIR99021 and BMP inhibitor LDN-193189.
We use \\ourmethod to investigate how the perturbation alters NMP differentiation, the relative bias toward somitic and neural tube fates, and the population dynamics of all three cell types.
Our analysis included $<<TLS_TREES>>$ TLS and $<<TLSC_TREES>>$ TLS$^{\\mathrm{CL}}$ lineage trees~\\cite{bolondi2024reconstructing} reconstructed from CRISPR--Cas9 lineage tracing, comprising $<<TLS_NMP>>$ NMPs, $<<TLS_NT>>$ neural tube cells, and $<<TLS_S>>$ somitic cells under the standard condition, and $<<TLSC_NMP>>$, $<<TLSC_NT>>$, and $<<TLSC_S>>$ cells of the corresponding types under perturbation.

We modeled NMP differentiation using a star-shaped transition graph with NMPs as the progenitor state and S and NT as terminal states, consistent with their established lineage relationships~\\cite{bolondi2024reconstructing}.
We restricted the lineage trees to these three states, which accounted for <<TLS_PERCENT>>\\% and <<TLSC_PERCENT>>\\% of sampled leaves in TLS and TLS$^{\\mathrm{CL}}$, respectively.
For the fits summarized here, the common sampling probability was fixed at $0.1$ across states in both conditions; it was not estimated separately between conditions.
Under the common-sampling assumption, the analysis focuses on transition probabilities, net growth rates, and scaled birth rates.
We compared prespecified, evenly spaced transition-time grids. For the 100-transition grid, the nominal spacing was $\\Delta t=0.01$; transition times were fixed rather than estimated jointly with the rates. These grid comparisons are a sensitivity analysis, not a complexity-penalized selection of the number of transition times or evidence that all fits converged.
To compare transition dynamics across spacings, we expressed the inferred transition probabilities as effective rates $q_{\\mathrm{NMP},i}=p_{\\mathrm{NMP},i}/\\Delta t$ for $i\\in\\{\\mathrm{S},\\mathrm{NT}\\}$, following the MTBD approximation.
<<TLS_CAVEAT>> <<TLSC_STATUS>>

Under standard TLS conditions, the 100-transition fit estimates preferential NMP differentiation toward neural tube (NT) despite similar net growth rates of the terminal states. The estimated net growth rates for NMPs, somites (S), and NT cells were $<<TLS_NMP_GROWTH>>$, $<<TLS_S_GROWTH>>$, and $<<TLS_NT_GROWTH>>$, respectively (per unit time), consistent with their role as a maintained progenitor population that supplies both lineages~\\cite{garriock2015lineage,bolondi2024reconstructing}.
Although NT cells outnumbered S cells in the sample ($<<TLS_NT>>$ versus $<<TLS_S>>$), their estimated net growth rates were similar. Instead, NMP differentiation favored NT, with an estimated effective transition-rate ratio $q_{\\mathrm{NMP},\\mathrm{NT}}/q_{\\mathrm{NMP},\\mathrm{S}}$ of $<<TLS_Q_RATIO>>$.
These estimates are consistent with preferential production of NT cells as one contributor to their greater abundance, illustrating how lineage-based inference can distinguish differentiation bias from differences in net growth.

WNT activation and BMP inhibition were associated with changes in differentiation bias and the relative growth of the two terminal cell states. In TLS$^{\\mathrm{CL}}$, NMPs retained the highest estimated net growth rate ($<<TLSC_NMP_GROWTH>>$), but S and NT dynamics diverged: relative to TLS, the estimated S net growth rate increased from $<<TLS_S_GROWTH>>$ to $<<TLSC_S_GROWTH>>$, whereas the NT net growth rate decreased from $<<TLS_NT_GROWTH>>$ to $<<TLSC_NT_GROWTH>>$.
NT cells retained a higher scaled birth rate than S cells (TLS: $<<TLS_NT_BIRTH_SCALED>>$ versus $<<TLS_S_BIRTH_SCALED>>$; TLS$^{\\mathrm{CL}}$: $<<TLSC_NT_BIRTH_SCALED>>$ versus $<<TLSC_S_BIRTH_SCALED>>$). The ratio $b_{\\mathrm{S}}/b_{\\mathrm{NT}}$ was $<<TLS_B_RATIO>>$ in TLS and $<<TLSC_B_RATIO>>$ in TLS$^{\\mathrm{CL}}$.
With sampling fixed equally across states, the fitted birth and net growth rates imply a higher NT death rate than S death rate in TLS$^{\\mathrm{CL}}$ ($<<TLSC_NT_DEATH>>$ versus $<<TLSC_S_DEATH>>$).
Differentiation also shifted from an NT bias toward an S bias, with $q_{\\mathrm{NMP},\\mathrm{NT}}/q_{\\mathrm{NMP},\\mathrm{S}}$ decreasing from $<<TLS_Q_RATIO>>$ to $<<TLSC_Q_RATIO>>$. This shift is consistent with the expansion of somitic tissue reported following WNT activation and BMP inhibition in TLS~\\cite{veenvliet2020mouse}.
Together, these results associate the perturbed phenotype with changes in both progenitor fate allocation and terminal-state net growth, despite a similar relative birth rate between the terminal states.

All rates and ratios in this generated text are 100-transition-grid point estimates, not medians. Cross-condition differences should be interpreted provisionally whenever the TLS fit is an incomplete checkpoint.'''

result_text = result_text.replace(chr(92) * 2, chr(92))

data_values = {
    "TLS_TREES": integer(tls_leaves["number_of_trees"]),
    "TLSC_TREES": integer(tlsc_leaves["number_of_trees"]),
    "TLS_NMP": integer(tls_leaves["NMPs"]),
    "TLS_NT": integer(tls_leaves["NeuralTube"]),
    "TLS_S": integer(tls_leaves["Somite"]),
    "TLSC_NMP": integer(tlsc_leaves["NMPs"]),
    "TLSC_NT": integer(tlsc_leaves["NeuralTube"]),
    "TLSC_S": integer(tlsc_leaves["Somite"]),
    "TLS_PERCENT": num(tls_leaves["three_state_leaf_percent"]),
    "TLSC_PERCENT": num(tlsc_leaves["three_state_leaf_percent"]),
    "TLS_CAVEAT": tls_fit_caveat,
    "TLSC_STATUS": tlsc_fit_status,
    "TLS_NMP_GROWTH": num(tls_nmp["net_growth_rate"]),
    "TLS_S_GROWTH": num(tls_s["net_growth_rate"]),
    "TLS_NT_GROWTH": num(tls_nt["net_growth_rate"]),
    "TLSC_NMP_GROWTH": num(tlsc_nmp["net_growth_rate"]),
    "TLSC_S_GROWTH": num(tlsc_s["net_growth_rate"]),
    "TLSC_NT_GROWTH": num(tlsc_nt["net_growth_rate"]),
    "TLS_Q_RATIO": num(tls_fit["q_NT_over_q_S"]),
    "TLSC_Q_RATIO": num(tlsc_fit["q_NT_over_q_S"]),
    "TLS_NT_BIRTH_SCALED": num(tls_nt["scaled_birth_rate"]),
    "TLS_S_BIRTH_SCALED": num(tls_s["scaled_birth_rate"]),
    "TLSC_NT_BIRTH_SCALED": num(tlsc_nt["scaled_birth_rate"]),
    "TLSC_S_BIRTH_SCALED": num(tlsc_s["scaled_birth_rate"]),
    "TLS_B_RATIO": num(tls_fit["b_S_over_b_NT"]),
    "TLSC_B_RATIO": num(tlsc_fit["b_S_over_b_NT"]),
    "TLSC_NT_DEATH": num(tlsc_nt["death_rate"]),
    "TLSC_S_DEATH": num(tlsc_s["death_rate"]),
}
for key, value in data_values.items():
    result_text = result_text.replace(f"<<{key}>>", value)
if "<<" in result_text or ">>" in result_text:
    raise ValueError("Unfilled result placeholder remains in generated text")

print(result_text)

We applied \ourmethod to study neuromesodermal progenitor (NMP) dynamics in mouse trunk-like structures (TLS), a stem-cell-derived model of early mammalian trunk development.
NMPs maintain a progenitor population while giving rise to neural tube and somitic cells, making TLS a useful system for investigating the balance between progenitor maintenance and differentiation~\cite{veenvliet2020mouse,bolondi2024reconstructing,sashittal2026inferring}.
We compared standard TLS with TLS$^{\mathrm{CL}}$, a perturbed condition treated with the WNT activator CHIR99021 and BMP inhibitor LDN-193189.
We use \ourmethod to investigate how the perturbation alters NMP differentiation, the relative bias toward somitic and neural tube fates, and the population dynamics of all three cell types.
Our analysis included $12$ TLS and $11$ TLS$^{\mathrm{CL}}$ lineage trees~\cite{bolondi2024reconstructing} reconstructed from CRISPR--Cas9 lineage tracing, comprising $378$ NMPs, $1,216$ neural tube cells, and $869$ 

In [ ]:
import math

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.path import Path as MplPath

node_colors = {
    "NMPs": "#8C8C8C",
    "NeuralTube": "#69B64F",
    "Somite": "#D85D91",
}
node_positions = {
    "NMPs": (0.50, 0.68),
    "NeuralTube": (0.20, 0.20),
    "Somite": (0.80, 0.20),
}

def parameter_value(dataset, state, column):
    rows = state_parameters.loc[
        (state_parameters["dataset"] == dataset)
        & (state_parameters["transition_times"] == 100)
        & (state_parameters["state"] == state),
        column,
    ]
    if len(rows) != 1:
        raise ValueError(f"Expected one 100-transition value for {dataset}/{state}/{column}")
    return float(rows.iloc[0])

def draw_directed_edge(ax, source, target, label, node_radius=0.16):
    dx = target[0] - source[0]
    dy = target[1] - source[1]
    distance = math.hypot(dx, dy)
    ux, uy = dx / distance, dy / distance
    start = (source[0] + node_radius * ux, source[1] + node_radius * uy)
    end = (target[0] - node_radius * ux, target[1] - node_radius * uy)
    ax.add_patch(FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=13,
        linewidth=1.6,
        color="#555555",
        zorder=1,
    ))
    midpoint = ((start[0] + end[0]) / 2, (start[1] + end[1]) / 2)
    normal = (-uy, ux)
    side = -1 if target[0] < source[0] else 1
    label_position = (
        midpoint[0] + side * 0.12 * normal[0],
        midpoint[1] + side * 0.12 * normal[1],
    )
    ax.text(
        *label_position,
        label,
        ha="center",
        va="center",
        fontsize=8,
        bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "edgecolor": "none", "alpha": 0.92},
        zorder=3,
    )

def draw_panel(ax, dataset, title):
    fit = fit_summary.loc[
        (fit_summary["dataset"] == dataset)
        & (fit_summary["transition_times"] == 100)
    ].iloc[0]
    ax.set_title(title, fontsize=15, pad=22)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.04, 1.18)
    ax.set_aspect("equal")
    ax.axis("off")

    q_rates = {
        "NeuralTube": float(fit["q_NMP_to_NT"]),
        "Somite": float(fit["q_NMP_to_S"]),
    }
    for target_state in ("NeuralTube", "Somite"):
        draw_directed_edge(
            ax,
            node_positions["NMPs"],
            node_positions[target_state],
            f"{('NT' if target_state == 'NeuralTube' else 'S')}: p/Δt = {q_rates[target_state]:.2f}",
        )

    # The self-loop denotes remaining in the NMP state; p/Δt is shown only for state changes.
    loop_vertices = [(0.42, 0.82), (0.28, 1.12), (0.72, 1.12), (0.58, 0.82)]
    loop_codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4]
    ax.add_patch(FancyArrowPatch(
        path=MplPath(loop_vertices, loop_codes),
        arrowstyle="-|>",
        mutation_scale=13,
        linewidth=1.4,
        color="#555555",
        zorder=1,
    ))

    display_names = {"NMPs": "NMPs", "NeuralTube": "Neural tube", "Somite": "Somite"}
    for state, (x, y) in node_positions.items():
        net_growth = parameter_value(dataset, state, "net_growth_rate")
        scaled_birth = parameter_value(dataset, state, "scaled_birth_rate")
        ax.add_patch(Circle(
            (x, y),
            radius=0.16,
            facecolor=node_colors[state],
            edgecolor="white",
            linewidth=1.2,
            alpha=0.82,
            zorder=2,
        ))
        ax.text(
            x,
            y,
            f"{display_names[state]}\nnet growth {net_growth:.2f}\nscaled birth {scaled_birth:.2f}",
            ha="center",
            va="center",
            fontsize=7.6,
            linespacing=1.15,
            color="#202020",
            zorder=4,
        )

fig, axes = plt.subplots(1, 2, figsize=(11.2, 6.2))
draw_panel(axes[0], "TLS", "TLS")
draw_panel(axes[1], "TLSC", r"TLS$^{\mathrm{CL}}$")
fig.text(
    0.5,
    0.035,
    "Node values: net growth = birth − death; scaled birth = birth × sampling. "
    "Edge values: transition probability / Δt per time unit. "
    "TLS is an iteration-4,360 checkpoint; TLS$^{\\mathrm{CL}}$ completed.",
    ha="center",
    va="bottom",
    fontsize=8,
)
fig.subplots_adjust(top=0.88, bottom=0.13, wspace=0.12)

figure_dir = repo_root / "data/troupe/figures"
figure_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_dir / "TLS_vs_TLSCL_transition_rates_100.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
fig.savefig(figure_path.with_suffix(".pdf"), bbox_inches="tight")
print(f"Saved {figure_path}")
print(f"Saved {figure_path.with_suffix('.pdf')}")
display(fig)
plt.close(fig)